### Autoencoders

The nonlinear counterpart to PCA (`unsupervised.ipynb`), and a genuinely useful addition to `anomaly-detection.ipynb`'s toolkit (reconstruction error as an anomaly score, not currently covered there).

#### 0. Core idea

A neural network trained to reconstruct its OWN input, output = input is the target, no external labels needed (self-supervised, same framing as Word2Vec in `nlp.ipynb`). The network is forced through a narrow BOTTLENECK layer, fewer dimensions than the input, so it cannot just copy the input through, it has to learn a compressed representation that captures enough information to reconstruct the input reasonably well.

Structure: encoder (input -> bottleneck, progressively smaller layers) -> bottleneck (the compressed representation) -> decoder (bottleneck -> reconstructed output, progressively larger layers, mirroring the encoder). Loss = reconstruction error, typically MSE between input and output (see `loss-functions.ipynb`).

#### 1. Comparison to PCA

PCA (`unsupervised.ipynb`) finds the best LINEAR low-dimensional projection, a closed-form solution via eigenvectors of the covariance matrix, fast, exact, no training loop. An autoencoder can learn a NONLINEAR compression instead (thanks to activation functions between layers, see `activation-functions.ipynb`), strictly more expressive, can capture curved/nonlinear structure in the data that no linear projection could, but at real cost: needs iterative training (no closed form), can overfit with too much capacity, and has no guarantee of finding the globally optimal compression the way PCA's eigendecomposition does.

Rule of thumb: start with PCA, it's cheap, exact, and interpretable (each component is a specific linear combination of original features). Reach for an autoencoder specifically when the data has genuine nonlinear structure PCA's linear projection can't capture, and you have enough data to train reliably without just memorizing it.

#### 2. Anomaly detection via reconstruction error

Train the autoencoder ONLY on normal data. It learns to compress and reconstruct patterns that look like normal data well, since that's all it ever saw. A genuinely anomalous input, one that doesn't resemble anything in the normal training distribution, reconstructs POORLY, the network has no learned compression scheme suited to it. High reconstruction error becomes the anomaly score, a real alternative (or complement) to `anomaly-detection.ipynb`'s Isolation Forest/One-Class SVM/LOF, particularly useful when the normal data has complex nonlinear structure those methods don't model well.

Worked illustration (using round numbers as if already trained, code below trains a real tiny one): a normal point x=[1,1] reconstructs as x_hat=[0.98,1.02], reconstruction error (MSE) = ((1-0.98)^2+(1-1.02)^2)/2 = 0.0004, tiny, good reconstruction. An anomalous point x=[5,5], far outside anything seen in training, reconstructs as x_hat=[1.2,1.3] (the network just falls back to something close to what it knows), error = ((5-1.2)^2+(5-1.3)^2)/2 = 14.07, orders of magnitude larger, a clear anomaly signal from the reconstruction gap alone.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))  # bottleneck=1
        self.decoder = nn.Sequential(nn.Linear(1, 4), nn.ReLU(), nn.Linear(4, 2))

    def forward(self, x):
        bottleneck = self.encoder(x)
        return self.decoder(bottleneck)

# train only on "normal" data, clustered near (1,1)
normal_data = torch.randn(200, 2) * 0.2 + torch.tensor([1.0, 1.0])

model = Autoencoder()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(300):
    optimizer.zero_grad()
    reconstructed = model(normal_data)
    loss = loss_fn(reconstructed, normal_data)
    loss.backward()
    optimizer.step()

print("final training reconstruction loss:", loss.item())

# test: a normal-looking point vs a clear outlier
test_points = torch.tensor([[1.0, 1.0], [1.1, 0.9], [5.0, 5.0]])
with torch.no_grad():
    reconstructed_test = model(test_points)
    errors = ((test_points - reconstructed_test) ** 2).mean(dim=1)

for point, err in zip(test_points, errors):
    print(f"point={point.tolist()}: reconstruction error={err.item():.4f}")

#### 3. Practical notes

Bottleneck size is the key hyperparameter, too large and the network can just learn to copy the input through with little real compression (defeats the purpose, and gives uselessly-low reconstruction error even on genuine anomalies), too small and it can't reconstruct even normal data well (high error everywhere, no useful separation between normal and anomalous). Same underfitting/overfitting tension as every other model in this series, just expressed through architecture width instead of a regularization parameter.

Variational Autoencoders (VAEs), briefly: add a probabilistic twist, the bottleneck represents a DISTRIBUTION (mean and variance) rather than a fixed point, enabling generation (sample from the bottleneck distribution, decode, get a new synthetic example), not just compression/reconstruction, out of scope for the worked treatment here but worth knowing the name and the one-line distinction (VAE generates, plain autoencoder compresses/reconstructs).